[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/claude-api-certified/notebooks/day-05-tool-use.ipynb#scrollTo=ea01fa02)

---
# Day 5 · Tool Use & Function Calling
**certified-journeys / claude-api-certified** · Schemas, tool_use blocks & multi-turn loops

> **Goal for today:** Build a multi-turn tool loop where Claude decides which functions to call, execute them, and feed results back — transforming Claude from a text generator into an autonomous actor.

In [ ]:
%pip install -q anthropic

In [ ]:
import os
import json
import anthropic

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except ImportError:
    pass

client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"

## Step 1 · Define a tool with a JSON Schema

A tool definition tells Claude what functions are available, what arguments they take, and what they return. Claude decides when and how to call them.

```
Tool definition structure:
  name        → what Claude calls the function
  description → when Claude should use this tool
  input_schema → JSON Schema for the arguments
```

In [ ]:
TOOLS = [
    {
        "name": "get_weather",
        "description": "Get the current weather for a city. Returns temperature in Celsius and conditions.",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "The city name, e.g. 'London' or 'Tokyo'"
                }
            },
            "required": ["city"]
        }
    },
    {
        "name": "calculate",
        "description": "Evaluate a mathematical expression and return the result. Use for arithmetic, not string manipulation.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "A Python-evaluable math expression, e.g. '(3 + 4) * 2'"
                }
            },
            "required": ["expression"]
        }
    }
]

print("Tools defined:", [t["name"] for t in TOOLS])

**What just happened?**
- **`description` is the most important field** — Claude reads it to decide when to call the tool.
- `input_schema` follows JSON Schema — use `required` for mandatory params and `description` on each property.
- **Never put secret logic in descriptions** — they're visible to the model and could be exploited.
- Tools can have 0 required parameters (e.g., `get_current_time()`).

## Step 2 · Implement the tool functions

The actual tool functions run in your Python code — Claude only sees the schema and the results you return.

In [ ]:
import random

# Mock weather data — in production, call a real weather API (OpenWeatherMap, etc.)
MOCK_WEATHER = {
    "london": {"temp_c": 12, "conditions": "Cloudy with light rain"},
    "tokyo": {"temp_c": 22, "conditions": "Clear and sunny"},
    "new york": {"temp_c": 8, "conditions": "Overcast"},
    "paris": {"temp_c": 15, "conditions": "Partly cloudy"},
    "sydney": {"temp_c": 28, "conditions": "Hot and sunny"},
}

def get_weather(city: str) -> dict:
    """Mock weather lookup. Production equivalent: call openweathermap.org API."""
    key = city.lower()
    if key in MOCK_WEATHER:
        return MOCK_WEATHER[key]
    # Return realistic-looking random data for unknown cities
    return {"temp_c": random.randint(-5, 35), "conditions": "Data unavailable — mock fallback"}


def calculate(expression: str) -> dict:
    """Safe math evaluator. Only allows numeric operations."""
    # Allowlist: only digits, operators, spaces, and parentheses
    allowed = set("0123456789+-*/().% ")
    if not all(c in allowed for c in expression):
        return {"error": "Expression contains disallowed characters"}
    try:
        result = eval(expression)  # safe because of the allowlist above
        return {"result": result}
    except Exception as e:
        return {"error": str(e)}


def dispatch_tool(tool_name: str, tool_input: dict) -> str:
    """Route a tool call to the correct function and return result as JSON string."""
    if tool_name == "get_weather":
        result = get_weather(**tool_input)
    elif tool_name == "calculate":
        result = calculate(**tool_input)
    else:
        result = {"error": f"Unknown tool: {tool_name}"}
    return json.dumps(result)


# Test the tools directly
print("Weather in Tokyo:", get_weather("Tokyo"))
print("Calculate (3 + 4) * 2:", calculate("(3 + 4) * 2"))

**What just happened?**
- **You control tool execution** — Claude suggests the call, but your code runs it and returns the result.
- **Input validation is critical** — the `calculate` allowlist prevents code injection.
- Return tool results as JSON strings — this is the format Claude expects in `tool_result` content blocks.
- `dispatch_tool()` is the integration point — this is where you'd add logging, retries, and error handling.

## Step 3 · Handle a tool_use response

When Claude wants to use a tool, `stop_reason` is `'tool_use'` and `response.content` contains a `ToolUseBlock`. You must execute the tool and send the result back.

In [ ]:
# Single tool call example
r = client.messages.create(
    model=MODEL,
    max_tokens=256,
    tools=TOOLS,
    messages=[{"role": "user", "content": "What's the weather like in London?"}]
)

print(f"Stop reason: {r.stop_reason}")
print(f"Content blocks: {len(r.content)}")

for block in r.content:
    print(f"  Block type: {block.type}")
    if block.type == "tool_use":
        print(f"  Tool: {block.name}")
        print(f"  Input: {block.input}")
        print(f"  ID: {block.id}")

**What just happened?**
- `stop_reason: 'tool_use'` signals that Claude wants to call a function before answering.
- **The `id` field** on `ToolUseBlock` must be referenced in your `tool_result` response.
- Claude may return **both text and tool_use blocks** — always iterate `r.content`, don't assume position.
- You must send the assistant's full `r.content` (not just the text) back in the next message.

## Step 4 · Build the multi-turn tool loop

The complete pattern: keep looping until `stop_reason == 'end_turn'`. Execute any tool calls and feed results back as `tool_result` blocks.

In [ ]:
def run_agent(user_message: str, max_iterations: int = 10) -> str:
    """Run a tool-using agent loop until Claude finishes or max_iterations is reached."""
    messages = [{"role": "user", "content": user_message}]
    iteration = 0

    while iteration < max_iterations:
        iteration += 1
        r = client.messages.create(
            model=MODEL,
            max_tokens=512,
            tools=TOOLS,
            messages=messages
        )

        # Append assistant's full response to history
        messages.append({"role": "assistant", "content": r.content})

        if r.stop_reason == "end_turn":
            # Extract final text response
            texts = [b.text for b in r.content if b.type == "text"]
            return "\n".join(texts)

        if r.stop_reason == "tool_use":
            # Execute all tool calls and collect results
            tool_results = []
            for block in r.content:
                if block.type == "tool_use":
                    print(f"  → Calling tool: {block.name}({block.input})")
                    result = dispatch_tool(block.name, block.input)
                    print(f"  ← Result: {result}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result
                    })

            # Send tool results back to Claude
            messages.append({"role": "user", "content": tool_results})
        else:
            break  # Unexpected stop_reason

    return "Agent reached max iterations without completing."


# Test with a multi-tool query
print("Query: What's the weather in Tokyo and what is 15% of 340?")
print("---")
answer = run_agent("What's the weather in Tokyo? Also, what is 15% of 340?")
print("\nFinal answer:")
print(answer)

**What just happened?**
- The loop runs until `stop_reason == 'end_turn'` — Claude indicates it has all the info it needs.
- **`max_iterations` is an error budget** — prevents infinite loops if something goes wrong.
- Claude can call multiple tools in a single turn — always iterate `r.content` to find all `tool_use` blocks.
- **`tool_use_id` must match exactly** the `id` from the `ToolUseBlock` — mismatches cause API errors.

## Step 5 · Force tool use and control tool choice

By default, Claude decides whether to use tools. You can override this with `tool_choice`.

In [ ]:
# Force Claude to use a specific tool
r = client.messages.create(
    model=MODEL,
    max_tokens=128,
    tools=TOOLS,
    tool_choice={"type": "tool", "name": "calculate"},  # force this tool
    messages=[{"role": "user", "content": "What is the square root of 144?"}]
)

print(f"Stop reason: {r.stop_reason}")
for block in r.content:
    if block.type == "tool_use":
        print(f"Tool called: {block.name}")
        print(f"Expression: {block.input['expression']}")

# Disable tools entirely
r2 = client.messages.create(
    model=MODEL,
    max_tokens=64,
    tools=TOOLS,
    tool_choice={"type": "none"},  # Claude cannot use any tools this turn
    messages=[{"role": "user", "content": "What is 2 + 2?"}]
)
print(f"\nWith tool_choice=none, stop reason: {r2.stop_reason}")
print(f"Response: {r2.content[0].text}")

**What just happened?**
- `{"type": "tool", "name": "calculate"}` — forces Claude to call that specific tool regardless of whether it would choose to.
- `{"type": "none"}` — Claude cannot use tools, even if they're listed. Useful for generating pure text responses.
- `{"type": "auto"}` (default) — Claude decides based on the question.
- **Forced tool use** is powerful for structured extraction pipelines where you always want the same tool called.

In [ ]:
# Challenge: Add a unit_convert tool to the agent
# Requirements:
# - Name: convert_units
# - Parameters: value (number), from_unit (str), to_unit (str)
# - Support: celsius<->fahrenheit, km<->miles, kg<->pounds
# - Add it to TOOLS and dispatch_tool(), then test with a query

def convert_units(value: float, from_unit: str, to_unit: str) -> dict:
    # Your solution here
    # Return: {"result": float, "from": str, "to": str}
    pass


# Test your function directly
result = convert_units(100, "celsius", "fahrenheit")
print("100 celsius in fahrenheit:", result)
assert result is not None, "convert_units should return a dict"
print("✓ Challenge passed!")

---
## Day 5 key concepts recap

| Concept | What to remember |
|---|---|
| Tool schema | `description` determines when Claude calls it — write it carefully |
| `stop_reason: 'tool_use'` | Claude wants to call a function — execute and return results |
| `tool_use_id` | Must match exactly when sending `tool_result` back |
| Tool loop | Keep looping until `end_turn` — add `max_iterations` guard |
| `tool_choice` | Force, auto, or disable tools per request |

> **Tip:** Tools transform Claude from a talker to a doer. Always handle the case where Claude calls multiple tools before returning a final answer.

---
## What's next
**Day 6** → RAG & Agentic Search: embed documents, build a vector store, and ground Claude's answers in your own data.

Mark Day 5 complete in your [tracker](../index.html).